# Final Project Phase 2 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 2 of the Final Project. Answer all statements addressed below as specified in the instructions for the project, covering all necessary details. Please be clear and concise in your answers. Each response should be at most 3 sentences. Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable. 
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or 
<a href="http://www.google.com">Google</a><br>

# Data Collection and Cleaning
You are required to provide data collection and cleaning for the three (3) minimum datasets. Create a function for each of the following sections that reads or scrapes data from a file or website, manipulate and cleans the parsed data, and writes the cleaned data into a new file. 

Make sure your data cleaning and manipulation process is not too simple. Performing complex manipulation and using modules not taught in class shows effort, which will increase the chance of receiving full credit.


## Data Sources
Include sources (as links) to your datasets. Add any additional data sources if needed. Clearly indicate if a data source is different from one submitted in your Phase I, as we will check that it satisfies the requirements.

### Note:
Note: In our Phase I doc, we originally said we would webscrape the BTS Baggage source but it turned out to be an embedded Excel table so bs4 was not making this work so we talked about it during out TA check-in and decided to switch sources. Original plan was to use bs4 on the FRED website where the CPI data lives and use that as the 4th source. We switched to web scrape it with bs4 as a primary source, and then analyze the BTS baggage fees and BTS consumer airfare data as downloaded data sets (XLSX and CSV files respectively), and then continue getting the EIA Oil prices using an API (JSON format) (no change to the plan here).


*   Downloaded Dataset #1 Source (Excel): 
    *   <a href="https://www.bts.gov/baggage-fees">Bureau of Transportation Aviation Statistics for Baggage Fees (All Years)</a><br>
    *   <a href="https://www.bts.gov/topics/airlines-and-airports/baggage-fees-airline-2025">BTS Baggage Fees Data (2025 + All Years) </a><br>
    *   Baggage Fees by Airline and Year (Excel file)
*   Downloaded Dataset #2 Source (CSV):
    *   <a href="https://www.transportation.gov/policy/aviation-policy/domestic-airline-consumer-airfare-report">BTS Domestic Consumer Airfare Data (Table 1)</a><br>
    *   <a href="https://data.transportation.gov/Aviation/Consumer-Airfare-Report-Table-2-Top-1-000-City-Pai/wqw2-rjgd/about_data">BTS Consumer Airfare Table 1 Data Specifics</a><br>
    *   Airfare prices and city pairs (CSV file)
*   Web Collection #1 Source (Web Scrapig/bs4):
    *   <a href="https://fred.stlouisfed.org/data/CPIAUCSL">FRED CPI Data</a><br>
    *   FRED (Govt economics database) CPI data website BeautifulSoup scraping
*   Web Collection #2 Source (API/JSON):
    *  <a href="https://api.eia.gov/v2/petroleum/pri/spt/data/">EIA API Endpoint</a><br>
    *  EIA Petroleum (WTI RWTC specifically)



In [1]:
import json
import requests
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime
from bs4 import BeautifulSoup as bs4

#### URLs, Paths & API Keys:
*   URL for Downloaded Dataset:
*   CPI Data URL (for bs4): CPI_URL
*   Get Request URL for Oil Data: EIA_BASE_URL
*   API Key for Oil Price Data: EIA_API_Key
*   Export path name for EIA data: oil_export_path
*   Export path name for raw CPI data: raw_cpi_export_path
*   Export path name for cleaned CPI data: clean_cpi_export_path
*   Excel path name for baggage fee data: baggage_path


In [2]:
# URLs, Paths & API Keys:

CPI_URL = "https://fred.stlouisfed.org/data/CPIAUCSL"
EIA_BASE_URL = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
EIA_API_KEY = "gc3m0emd44qaMHTNQxwmLqupbzdCtHHB2G6aOCNl"
oil_export_path = "eia_wti_quarterly.csv"
raw_cpi_export_path = "cpi_raw.csv"
clean_cpi_export_path = "cpi_clean.csv"
baggage_path = 'BaggageFees.xlsx'


pd.set_option('future.no_silent_downcasting', True) # this is to silence errors I was getting with pandas versions 

# Note: you can also change base year explicitly in the CPI function with optional paramter base_year (default = 2025)

## Downloaded Dataset Requirement

Fill in the predefined functions with your data scraping/parsing code. You may modify/rename each function as you seem fit, but you must provide at least 3 separate functions that clean each of your required datasets.


In [3]:
def row_contains_keyword(row, keywords):
    #Check if a row contains keywords, which are likely column names.
    vals = [("" if pd.isna(v) else str(v).strip().lower()) for v in row.values]
    return any(word.lower() in v for v in vals for word in keywords)

def find_header_row(df):
    #Finds the row number that's probably the header
    keywords = ['rank', 'airline', '1Q', '2Q', '3Q', '4Q']
    for i, row in df.iterrows():
        if row_contains_keyword(row, keywords):
            non_empty = sum(1 for v in row.values if not pd.isna(v) and str(v).strip() != '')
            if non_empty >= 2:
                return i
    return None

def clean_dataframe(df):
    # Strip column names
    df.columns = [("" if pd.isna(x) else str(x).strip()) for x in df.columns]

    # Remove empty columns
    df = df.dropna(axis=1, how='all')

    # Find airline column
    key_col = None
    for col in df.columns:
        if 'airline' in str(col).strip().lower():
            key_col = col
            break

    # Remove rows with empty airline column
    if key_col:
        df = df[df[key_col].notna()]
        df.reset_index(drop=True, inplace=True)

    return df

# Dictionary to store all cleaned DataFrames
all_sheets = {}

baggage = pd.ExcelFile(baggage_path)

# Filter sheets from 2007 to 2025
sheets = [s for s in baggage.sheet_names if re.match(r'^(2007|2008|2009|201\d|202[0-5])$', s)]

for sheet_name in sheets:
    # Read to get row that's probably header
    df = pd.read_excel(baggage, sheet_name=sheet_name, header=None, dtype=object)
    
    header_row = find_header_row(df)
    
    # Now look at it now we know the header
    df = pd.read_excel(baggage, sheet_name=sheet_name, header=header_row, dtype=object)
    
    df = clean_dataframe(df)
    
    all_sheets[sheet_name] = df
    print(sheet_name)
    print(df.head(), "\n")


for sheet_name in all_sheets:
    if(sheet_name == '2025'):
        break;
    df = all_sheets[sheet_name]
    df = df.dropna() 
    all_sheets[sheet_name] = df.reset_index(drop=True)

for sheet_name in all_sheets: 
    all_sheets[sheet_name].drop('Rank', axis=1, errors='ignore', inplace=True)
    print(sheet_name)
    print(all_sheets[sheet_name])

for sheet_name, df in all_sheets.items():
    cols = [col for col in df.columns if str(col).strip().upper() in ['1Q', '2Q', '3Q', '4Q','Full Year']]
    
    for col in cols:
        df[col] = df[col].replace(['-', None], 0).astype(float)


for sheet_name in sheets:
    print(sheet_name)
    print(all_sheets[sheet_name].head())

############ Function Call ############


2007
  Rank               Airline     1Q     2Q     3Q     4Q Full Year
0    1   American Airlines    28829  30015  31346  34348    124538
1    2     Delta Air Lines    20343  22447  25035  28721     96546
2    3       United Airlines  12045  13410  14829  12718     53002
3    4  Continental Airlines  10715  11045  10565  10519     42844
4    5  Northwest Airlines     8339   8956   9897  10309     37501 

2008
  Rank               Airline     1Q     2Q     3Q      4Q Full Year
0    1   American Airlines    32959  37101  94075  113856    277991
1    2         US Airways      7478  17917  67928   93759    187082
2    3     Delta Air Lines    26571  42461  47489   60542    177063
3    4       United Airlines  12219  19721  42283   58771    132994
4    5  Northwest Airlines     9641  15685  32695   63578    121599 

2009
  Rank               Airline      1Q      2Q      3Q      4Q Full Year
0    1     Delta Air Lines    102838  118356  129465  131060    481719
1    2   American Airlines   

## Additional Downloaded Data Set

In [4]:
airfares = pd.read_csv("ConsumerAirfares.csv")

# Drops columns that are duplicates or unnecessary
airfares = airfares.drop(["table_1_flag","Geocoded_City1","Geocoded_City2","Geocoded_City1 (city)","Geocoded_City2 (city)"],axis=1)

### Inconsistency: Fill necessary columns with missing values that have NaN values with 0
airfares["lf_ms"] = airfares["lf_ms"].fillna(0)
airfares["carrier_low"] = airfares["carrier_low"].fillna(0)
airfares["fare_low"] = airfares["fare_low"].fillna(0)
airfares = airfares.dropna(axis=1)

### Inconsistency: citymarketid_1 was a object with commas whereas citymarketid_2 was a column of ints, changed citymarketid_1 to be column of ints
airfares["citymarketid_1"] = airfares["citymarketid_1"].str.replace(',','').astype(int)

### Somewhat inconsistencies: changed passengers, fare, fare_large, fare_low to be integers or floats rounded to two decimal points to make calculations easier in the future
airfares["passengers"] = airfares["passengers"].astype(str).str.replace(',', '').astype(int)
airfares["fare"] = round(airfares["fare"].astype(str).str.replace('$','').astype(float),2)
airfares["fare_lg"] = round(airfares["fare_lg"].astype(str).str.replace('$','').astype(float),2)
airfares["fare_low"] = round(airfares["fare_low"].astype(str).str.replace('$','').astype(float),2)

airfares

,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,nsmiles,passengers,fare,carrier_lg,large_ms,fare_lg,carrier_low,lf_ms,fare_low
0,2025,2,32467,31703,"Miami, FL (Metropolitan Area)","New York City, NY (Metropolitan Area)",1118,17955,208.52,B6,0.2551,191.48,B6,0.2551,191.48
1,2025,2,32575,32457,"Los Angeles, CA (Metropolitan Area)","San Francisco, CA (Metropolitan Area)",372,17310,157.68,WN,0.5006,169.03,AS,0.1193,140.59
2,2025,2,32575,31703,"Los Angeles, CA (Metropolitan Area)","New York City, NY (Metropolitan Area)",2510,13648,430.38,DL,0.2535,526.21,B6,0.2272,365.63
3,2025,2,31703,31454,"New York City, NY (Metropolitan Area)","Orlando, FL",989,12627,186.50,B6,0.3735,186.10,B6,0.3735,186.10
4,2025,2,30977,31703,"Chicago, IL","New York City, NY (Metropolitan Area)",773,11284,221.33,UA,0.4328,238.62,AA,0.2426,217.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118030,1996,1,33495,31454,"New Orleans, LA","Orlando, FL",550,111,138.88,DL,0.2600,174.49,NW,0.1400,112.40
118031,1996,1,30647,31995,"Cleveland, OH (Metropolitan Area)","Greensboro/High Point, NC",381,111,199.92,CO,0.8400,193.76,CO,0.8400,193.76
118032,1996,1,30158,31454,"Atlantic City, NJ","Orlando, FL",852,111,95.23,NK,0.9300,91.49,NK,0.9300,91.49
118033,1996,1,33244,33495,"Memphis, TN","New Orleans, LA",349,110,201.52,NW,0.8800,204.78,J7,0.0400,121.56


## Web Collection Requirement \#1


In [7]:
# Note: Pylance errors because the skeleton template puts the helper functions below this block, so make sure to run those before this.

# HTML notes: th holds date, td holds value
def get_quarterly_cpi(raw_path=raw_cpi_export_path,clean_path=clean_cpi_export_path,data_url=CPI_URL,base_year=2025,silence=False):
    html = requests.get(data_url).text
    soup = bs4(html, "html.parser")
    table = soup.find("table", id="data-table-observations")
    if not table: # for debugging
        raise ValueError("CPI table not found; check HTML structure")

    rows = []
    tbody = table.find("tbody")
    for tr in tbody.find_all("tr"):
        th = tr.find("th")
        tds = tr.find_all("td")
        if not th or not tds:
            continue # skip irrelavant elements
        rows.append((th.get_text().strip(), tds[0].get_text().strip()))

    cpi = pd.DataFrame(rows, columns=["date","value"])
    cpi["date"]  = pd.to_datetime(cpi["date"])
    cpi["value"] = pd.to_numeric(cpi["value"])
    cpi = cpi.dropna().sort_values("date").reset_index(drop=True) # dropna not really needed but I figured it would make it semi future proof
    
    if silence != False:
        save_csv(cpi,raw_path,silence=silence) # store raw data w/ no "Rewrote..." message for nesting in another function
    else:
        save_csv(cpi, raw_path) # store raw data with explicit rewrite message (default silence=False)

    cpi["qstart"] = cpi["date"].dt.to_period("Q").dt.start_time
    cpi_q = (cpi.groupby("qstart",as_index=False)["value"]
               .mean()
               .rename(columns={"qstart":"quarter_start_date","value":"cpi_index"}))
    cpi_q["quarter"] = cpi_q["quarter_start_date"].apply(to_quarter_label)

    q = cpi_q["quarter"].astype(str)
    m = q.str.startswith(str(base_year))
    base = cpi_q[m].iloc[-1] if m.any() else cpi_q.iloc[-1]

    # using .copy() to start clean, not technically needed though. Also good for splitting nom vs real $.
    out = cpi_q[["quarter","cpi_index"]].copy()
    out["cpi_base_year"] = base_year
    out["cpi_base_value"] = float(base["cpi_index"])
    out["cpi_base_quarter_used"] = str(base["quarter"])

    if silence != False:
        save_csv(out, clean_path,silence=silence) # store raw data w/ no "Rewrote..." message for nesting in another function
    else:
        save_csv(out, clean_path) # store clean data

    return out


############ Function Call ############
get_quarterly_cpi()


rewrote cpi_raw.csv
rewrote cpi_clean.csv


,quarter,cpi_index,cpi_base_year,cpi_base_value,cpi_base_quarter_used
0,1947Q1,21.700000,2025,323.288,2025Q3
1,1947Q2,22.010000,2025,323.288,2025Q3
2,1947Q3,22.490000,2025,323.288,2025Q3
3,1947Q4,23.126667,2025,323.288,2025Q3
4,1948Q1,23.616667,2025,323.288,2025Q3
...,...,...,...,...,...
310,2024Q3,314.182667,2025,323.288,2025Q3
311,2024Q4,316.538667,2025,323.288,2025Q3
312,2025Q1,319.492000,2025,323.288,2025Q3
313,2025Q2,320.800333,2025,323.288,2025Q3


## Web Collection Requirement \#2

In [8]:
# CPI must successfully run before this one as CPI is used for initial normalization to adjust for inflation (use 2025 $; base year in cpi function)
# Note: Same Pylance errors as CPI due to skeleton template code block placement

# Note: Columns are ['period', 'duoarea', 'area-name', 'product', 'product-name', 'process', 'process-name', 'series', 'series-description', 'value', 'units']
# Note: nested get_quarterly_cpi() call to ensure CPI data is fresh each time this function is run. They can be run seperately though (set cpi_df=cpi_df)

def get_eia_data(cpi_df=get_quarterly_cpi(silence=True), base=EIA_BASE_URL, api_key=EIA_API_KEY,export_path=oil_export_path):
    params = {
        "api_key": api_key,
        "frequency": "monthly", # quartlery not available so this is one of the inconsistencies
        "data[0]": "value", # name of key metric in API so have to set it explicitly
        "length": 5000,  # need to expand page size because data runs long
        "start": "2000-01-01" # can change in Phase III based on what data bottlenecks on dates
    }
    r = requests.get(base, params=params)
    parsed = json.loads(r.text)

    rows = []
    if isinstance(parsed, dict): # checks object is a dict format, throws error print message if format changes
        rows = parsed["response"]["data"]

    eia = pd.DataFrame(rows)

    if eia.empty:
        return "Not working, data is empty." # debug if data is empty

    eia = eia[eia["series"].eq("RWTC")] # mask Boolean indexing to get only WTI spot prices
    # Note: RWTC is the specific WTI provided spot price name that seems to be best for aviation purposes based on what we researched


    eia["period"] = pd.to_datetime(eia["period"])
    eia["value"] = pd.to_numeric(eia.get("value", np.nan))
    eia = eia.dropna(subset=["period", "value"]) # drops any rows with NaN in period or value, but we didn't see any

    # monthly -> quarterly mean
    eia["qstart"] = eia["period"].dt.to_period("Q").dt.start_time
    wti_q = (
        eia.groupby("qstart", as_index=False)["value"]
           .mean()
           .rename(columns={"qstart": "quarter_start_date", "value": "wti_usd_nominal"})
    )
    wti_q["quarter"] = wti_q["quarter_start_date"].apply(to_quarter_label)

    # inflation adjust with CPI already built
    cpi_small = cpi_df[["quarter", "cpi_index", "cpi_base_value"]].copy()
    merged = wti_q.merge(cpi_small, on="quarter", how="left")
    
    # Math is based on base year (from CPI function call)
    merged["wti_usd_real"] = merged["wti_usd_nominal"] * (merged["cpi_base_value"] / merged["cpi_index"])

    eia_clean = merged[["quarter", "wti_usd_nominal", "wti_usd_real"]].sort_values("quarter").reset_index(drop=True)
    save_csv(eia_clean, export_path) # store inflation-adjusted oil price data
    return eia_clean

############ Function Call ############
get_eia_data()

rewrote eia_wti_quarterly.csv


,quarter,wti_usd_nominal,wti_usd_real
0,2000Q1,28.823333,54.780939
1,2000Q2,28.776667,54.266873
2,2000Q3,31.613333,59.076366
3,2000Q4,31.990000,59.357087
4,2001Q1,28.816667,52.962379
...,...,...,...
99,2024Q4,70.686667,72.193869
100,2025Q1,71.836667,72.690184
101,2025Q2,64.626667,65.127818
102,2025Q3,65.736667,65.736667


## Additional Dataset Parsing/Cleaning Functions

Write any supplemental (optional) functions here.

In [6]:
# formatting dates, need to use universally
# date format: YYYYQn (e.g. 2023Q1, 2025Q2)
def to_quarter_label(ts):
    q = ((ts.month - 1) // 3) + 1 # map month to Q when divide by 3, but add 1 to make Q0 Q1
    return f"{ts.year}Q{q}"

# save dataframe to csv with an explicit mention of the path
def save_csv(df, path, silence=False):
    df.to_csv(path, index=False)
    if silence != True:
        print(f"rewrote {path}") # to help debugging

    
############ Function Call ############
# for to_quarter_label, single and pandas examples:
print(to_quarter_label(datetime(2025, 2, 15)))
dates = pd.to_datetime(["2024-01-10", "2024-05-20", "2024-12-31"])
print(dates.map(to_quarter_label).tolist())

# no test for csv function call but ->
# Note: We added the CSV silence feature to better our debugging and minimize unnecessary messages being printed on function calls when we nested the CPI function in the EIA function

2025Q1
['2024Q1', '2024Q2', '2024Q4']


In [ ]:
# Define further extra source functions as necessary

# Inconsistencies

For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. <b>Missing values in low-fare carrier columns (Consumer Airfares dataset):</b> The lf_ms, carrier_low, and fare_low columns contained NaN values where low-fare carrier data was unavailable for certain routes, which we identified through manual queries/scripts to evaluate the airfare csv file. We filled these missing values with 0 to indicate the absence of a low-fare competitor, preventing errors in downstream calculations and ensuring complete records for all city pairs.



2. <b>Inconsistent data types for city market identifiers (Consumer Airfares dataset):</b> The citymarketid_1 column was stored as object type with comma separators (e.g. "32,467"), while citymarketid_2 was stored as integers without formatting, which we found through manual queries/scripts to evaluate the CSV file. We removed commas from citymarketid_1 and explictly casted those values to integer type to enable proper and clean joins, numerical comparisons/arithmetic, and consistent handling of both city identifier columns.


3. <b>Currency and numeric formatting preventing calculations (Consumer Airfares dataset):</b> The passengers, fare, fare_lg, and fare_low columns contained formatting characters including commas and dollar signs that prevented numeric operations. We removed these characters and converted passengers to integers and all fare columns to floats rounded to two decimal places, which is the standard precision for currency values and enables mathematical analysis.

4. <b>Inconsistent header positioning across Excel sheets (Baggage Fees dataset):</b> The baggage fee Excel file contained different header row positions across years (2007-2025), with some sheets having metadata or blank rows above the actual column headers. We implemented dynamic header detection using keyword matching to identify the correct header row for each sheet, ensuring consistent parsing across all annual tabs.

5. <b>Temporal granularity mismatch for oil price data (EIA API dataset):</b> The EIA API provides oil prices only at a monthly frequency, while our airfare and baggage datasets use quarterly intervals. We used a quarter conversion function to aggregate monthly WTI RWTC petroleum prices into quarterly averages, enabling consistent time alignment and merging across datasets.